In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import pennylane as qml
from pennylane import numpy as np
from pennylane.optimize import NesterovMomentumOptimizer
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
print(df.shape)
df.head()


: 

In [ ]:
print(data.target_names)
print(df['target'].value_counts())


In [ ]:
print(df.isnull().sum().sum())

In [ ]:
print(data.feature_names)
df.describe()

In [ ]:
malignant = df[df['target'] == 0]['mean radius']
benign = df[df['target'] == 1]['mean radius']

plt.hist(malignant, bins=30, alpha=0.5, label='malignant', color='red')
plt.hist(benign, bins=30, alpha=0.5, label='benign', color='green')
plt.legend()
plt.xlabel('mean radius')
plt.show()
                        

In [ ]:
corr = df.drop('target', axis=1).corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='RdYlGn', center=0)
plt.show()

In [ ]:
scaler = StandardScaler()
X = df.drop('target', axis=1).values
y = df['target'].values

X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_scaled)

print(pca.explained_variance_ratio_)
print(f"Total variance kept: {pca.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
n_layers = 2
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

a = 0.36
b = 0.64

B = np.array(
    [[a,  0, 0,  b],
     [0, -a, b,  0],
     [0,  b, a,  0],
     [b,  0, 0, -a]]
)

weights_shape = qml.StronglyEntanglingLayers.shape(n_layers=n_layers, n_wires=n_qubits)
weights = np.random.uniform(0, np.pi, weights_shape)

x = np.random.uniform(0, np.pi, n_qubits)

@qml.qnode(dev)
def full_pipeline(x, weights):
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))

print(full_pipeline(x, weights))

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from pennylane.optimize import AdamOptimizer
from pennylane import numpy as pnp

data = load_breast_cancer()
X = data.data
y = data.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_scaled)

X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape)

def classifier(weights, bias, x):
    return full_pipeline(x, weights) + bias

def square_loss(labels, predictions):
    preds = qml.math.stack(predictions)
    return qml.math.sum((labels - preds) ** 2) / len(labels)

def cost(weights, bias, X, y):
    preds = [classifier(weights, bias, x) for x in X]
    return square_loss(y, preds)

def accuracy(labels, weights, bias, X):
    preds = np.sign(np.array([float(classifier(weights, bias, x)) for x in X]))
    return np.mean(np.array(labels) == preds)

label = 1.0
prediction = 0.40

opt = AdamOptimizer(stepsize=0.1)
weights = pnp.random.uniform(0, pnp.pi, weights_shape, requires_grad=True)
bias = pnp.array(0.0, requires_grad=True)

print(weights.shape)
print(bias)

X_sample = pnp.array(X_train[:5], requires_grad=False)
y_sample = pnp.array(y_train[:5], requires_grad=False)
print(cost(weights, bias, X_sample, y_sample))

In [ ]:
#STRONGLY ENTANGLING LAYERS
import numpy as np
X_tr = pnp.array(X_train, requires_grad=False)
Y_tr = pnp.array(2 * y_train - 1, requires_grad=False)
X_te = pnp.array(X_test, requires_grad=False)
Y_te = pnp.array(2 * y_test - 1, requires_grad=False)

n_layers = 10
weights_shape = qml.StronglyEntanglingLayers.shape(n_layers=n_layers, n_wires=n_qubits)

pnp.random.seed(42)
weights = pnp.array(np.random.uniform(0, np.pi, weights_shape), requires_grad=True)
bias = pnp.array(0.0, requires_grad=True)
opt = AdamOptimizer(0.08)



batch_size = 15
n_epochs = 5
best_acc = 0.0
best_cost = 0.0

for epoch in range(n_epochs):
    idx = pnp.random.randint(0, len(X_tr), (batch_size,))
    result = opt.step(cost, weights, bias, X_tr[idx], Y_tr[idx])
    weights, bias = result[0], result[1]
    c = cost(weights, bias, X_tr, Y_tr)
    acc = accuracy(Y_te, weights, bias, X_te)
    print(f"Epoch {epoch+1:3d} | Cost: {c:.4f} | Test Acc: {acc*100:.1f}%")
    print(f"Weights: {weights.reshape(10, 12)}")
    print(f"Bias: {bias:.4f}")
    if acc > best_acc:
        best_acc = acc
    if c < best_cost or best_cost == 0.0:
        best_cost = c
print(f"Best Test Accuracy: {best_acc*100:.1f}%")
print(f"Best Test Cost: {best_cost:.4f}")

In [ ]:
#BASIC ENTANGLEMENT
@qml.qnode(dev)
def full_pipeline(x, weights):
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))

X_tr = pnp.array(X_train, requires_grad=False)
Y_tr = pnp.array(2 * y_train - 1, requires_grad=False)
X_te = pnp.array(X_test, requires_grad=False)
Y_te = pnp.array(2 * y_test - 1, requires_grad=False)

n_layers = 10
weights_shape = qml.BasicEntanglerLayers.shape(n_layers=n_layers, n_wires=n_qubits)

pnp.random.seed(42)
weights = pnp.array(np.random.uniform(0, np.pi, weights_shape), requires_grad=True)
bias = pnp.array(0.0, requires_grad=True)
opt = AdamOptimizer(0.08)

batch_size = 15
n_epochs = 5
best_acc = 0.0
best_cost = 0.0

for epoch in range(n_epochs):
    idx = pnp.random.randint(0, len(X_tr), (batch_size,))
    result = opt.step(cost, weights, bias, X_tr[idx], Y_tr[idx])
    weights, bias = result[0], result[1]
    c = cost(weights, bias, X_tr, Y_tr)
    acc = accuracy(Y_te, weights, bias, X_te)
    print(f"Epoch {epoch+1:3d} | Cost: {c:.4f} | Test Acc: {acc*100:.1f}%")
    print(f"Weights: {weights.reshape(8, 5)}")
    print(f"Bias: {bias:.4f}")
    if acc > best_acc:
        best_acc = acc
    if c < best_cost or best_cost == 0.0:
        best_cost = c
print(f"Best Test Accuracy: {best_acc*100:.1f}%")
print(f"Best Test Cost: {best_cost:.4f}")